In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pywt

In [2]:
def wavelet_hash(image):
    # Đọc ảnh nếu đầu vào là đường dẫn
    if isinstance(image, str):
        image = cv2.imread(image)

    # Kiểm tra ảnh hợp lệ
    if image is None:
        raise ValueError("Ảnh không hợp lệ hoặc không đọc được!")

    # Chuyển ảnh sang ảnh xám
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Chuẩn hóa kích thước ảnh
    image = cv2.resize(image, (16, 16))

    # Biến đổi Wavelet Haar
    coeffs = pywt.dwt2(image.astype(np.float32), 'haar')
    LL, (LH, HL, HH) = coeffs

    # Lấy trung vị của hệ số xấp xỉ
    median = np.median(LL)

    # Tạo mã nhị phân 64 bit
    bits = LL > median

    # Chuyển thành chuỗi 16 ký tự hexadecimal
    hash_value = format(
        int(''.join('1' if b else '0' for b in bits.flatten()), 2),
        '016x'
    )

    return hash_value

In [7]:
image_path = r'C:\Users\adu\Computer-Vision-Group-2\lab_4\data\input\anh_dem.jpg'

image = cv2.imread(image_path)

if image is None:
    print("Không đọc được ảnh. Hãy kiểm tra đường dẫn.")
else:
    # Ảnh giống hệt
    same_image = image.copy()

    # Ảnh thay đổi nhẹ: giảm độ sáng
    slight_image = cv2.convertScaleAbs(image, alpha=0.95, beta=0)

    # Ảnh khác: thay bằng đường dẫn ảnh khác của bạn
    other_path = r'C:\Users\adu\Computer-Vision-Group-2\lab_4\data\input\rung_cay.jpg'
    other_image = cv2.imread(other_path)

    print("Đã đọc ảnh gốc:", image.shape)
    print("Đọc được ảnh khác:", other_image is not None)

Đã đọc ảnh gốc: (4088, 2730, 3)
Đọc được ảnh khác: True


### Tính hash 

In [8]:
if image is not None and other_image is not None:
    hash1 = wavelet_hash(image)
    hash2 = wavelet_hash(same_image)
    hash3 = wavelet_hash(slight_image)
    hash4 = wavelet_hash(other_image)

    print("Ảnh gốc:       ", hash1)
    print("Ảnh giống hệt: ", hash2)
    print("Ảnh đổi nhẹ:   ", hash3)
    print("Ảnh khác:      ", hash4)

    print("\nĐộ dài hash:", len(hash1))

Ảnh gốc:        20283e7e7e7e1c1c
Ảnh giống hệt:  20283e7e7e7e1c1c
Ảnh đổi nhẹ:    20283e7e7e7e1c1c
Ảnh khác:       fefd7d1f09012820

Độ dài hash: 16


### So sánh khoảng cách hash 

Khoảng cách Hamming là số bit khác nhau giữa hai mã hash. Khoảng cách càng nhỏ thì hai hash càng giống nhau.

In [9]:
def hamming_distance(hash1, hash2):
    a = int(hash1, 16)
    b = int(hash2, 16)
    return bin(a ^ b).count('1')


if image is not None and other_image is not None:
    print("Ảnh gốc và ảnh giống hệt:",
          hamming_distance(hash1, hash2))

    print("Ảnh gốc và ảnh đổi nhẹ:",
          hamming_distance(hash1, hash3))

    print("Ảnh gốc và ảnh khác:",
          hamming_distance(hash1, hash4))

Ảnh gốc và ảnh giống hệt: 0
Ảnh gốc và ảnh đổi nhẹ: 0
Ảnh gốc và ảnh khác: 37


Kết quả mong đợi:
- Ảnh giống hệt: khoảng cách bằng 0.
- Ảnh thay đổi nhẹ: thường có khoảng cách nhỏ, nhưng không đảm bảo luôn nhỏ với mọi thay đổi.
- Ảnh khác nhau: thường có khoảng cách lớn hơn, nhưng còn phụ thuộc nội dung ảnh.

### Nhận xét về Wavelet Hash

Wavelet Hash sử dụng biến đổi Wavelet Haar để trích xuất hệ số xấp xỉ của ảnh. Các hệ số này được so sánh với giá trị trung vị để tạo mã nhị phân có độ dài cố định.

Hàm `wavelet_hash()` nhận ảnh hoặc đường dẫn ảnh, kiểm tra dữ liệu đầu vào và trả về mã hash gồm 16 ký tự hexadecimal, tương ứng 64 bit.

Kết quả thử nghiệm được đánh giá bằng khoảng cách Hamming. Hai ảnh giống hệt nhau có hash giống nhau và khoảng cách bằng 0. Ảnh thay đổi nhẹ thường có hash gần nhau, trong khi ảnh khác nhau thường có khoảng cách lớn hơn.

Phương pháp này giúp so sánh đặc trưng tổng quát của ảnh mà không cần so sánh trực tiếp từng pixel. Tuy nhiên, kết quả còn phụ thuộc vào cách chuẩn hóa ảnh và mức độ thay đổi của hình ảnh.